# From NOAA to Your Browser: Building a Real-Time Weather Radar

### Robotics Reading Group | April 2026

---

**What we built:** A web-based weather radar viewer that pulls live MRMS (Multi-Radar Multi-Sensor) data from NOAA's public AWS bucket and renders it over a map of the continental US -- with 3D volume rendering, smooth animation, and motion-compensated interpolation between frames.

**Tech stack:** Python (FastAPI) backend, vanilla JS frontend with MapLibre GL + three.js + GLSL shaders.

**Why it's interesting for this group:** Real-time sensor fusion, sparse data handling, GPU rendering pipelines, and signal processing -- concepts that transfer directly to robotics perception stacks.

In [ ]:
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, ArrowStyle
from matplotlib.collections import LineCollection
import numpy as np

plt.rcParams.update({
    'figure.facecolor': '#0f1729',
    'axes.facecolor': '#0f1729',
    'text.color': '#e0e6f0',
    'axes.labelcolor': '#e0e6f0',
    'xtick.color': '#8892a8',
    'ytick.color': '#8892a8',
    'axes.edgecolor': '#2a3550',
    'grid.color': '#1e2a45',
    'font.family': 'sans-serif',
    'font.size': 12,
})

ACCENT = '#4fc3f7'
ACCENT2 = '#81c784'
ACCENT3 = '#ffb74d'
WARN = '#ef5350'
DIM = '#5a6680'

print('Setup complete.')

---
## Slide 1: What is MRMS?

**Multi-Radar Multi-Sensor (MRMS)** is NOAA's flagship radar mosaic system.

- Fuses data from **180+ NEXRAD radars** across the US into a single, seamless product
- Updated every **2 minutes** -- among the fastest operational radar products in the world
- Covers all of CONUS (Continental US) on a regular **0.01° lat/lon grid** (3500 x 7000 = 24.5 million points)
- Published to a **free, public S3 bucket** on AWS (`noaa-mrms-pds`) -- no credentials, no API keys
- Provides **tilt-level reflectivity** at 8 elevation angles, giving both 2D and 3D views from one data source

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

# CONUS bounds (approximate MRMS coverage)
lon_min, lon_max = -130, -60
lat_min, lat_max = 20, 55

# Simplified US state boundaries (major outline points)
us_outline = np.array([
    [-124.7, 48.4], [-123.0, 48.4], [-122.8, 47.0], [-124.5, 46.3],
    [-124.0, 42.0], [-120.0, 42.0], [-120.0, 39.0], [-117.0, 36.9],
    [-114.6, 35.0], [-114.6, 32.7], [-111.0, 31.3], [-108.2, 31.8],
    [-106.6, 31.8], [-103.0, 29.0], [-100.0, 28.0], [-97.2, 25.9],
    [-97.2, 27.5], [-94.0, 29.5], [-89.6, 29.0], [-89.0, 30.2],
    [-85.0, 29.7], [-82.5, 27.5], [-80.0, 25.2], [-80.1, 28.5],
    [-81.0, 31.0], [-78.5, 33.9], [-75.5, 35.5], [-75.0, 38.0],
    [-74.0, 39.7], [-73.9, 40.5], [-71.8, 41.3], [-70.0, 41.8],
    [-69.8, 43.8], [-67.0, 44.8], [-67.0, 47.2], [-69.0, 47.4],
    [-72.0, 45.0], [-75.0, 45.0], [-79.0, 43.2], [-82.5, 41.7],
    [-83.0, 42.0], [-82.5, 46.0], [-84.5, 46.5], [-88.0, 48.0],
    [-89.5, 48.0], [-94.7, 49.0], [-104.0, 49.0], [-117.0, 49.0],
    [-122.8, 49.0], [-124.7, 48.4]
])

# Draw MRMS grid coverage area
grid_rect = mpatches.Rectangle((lon_min, lat_min), lon_max - lon_min, lat_max - lat_min,
                                linewidth=1.5, edgecolor=ACCENT, facecolor=ACCENT,
                                alpha=0.08, linestyle='--', zorder=1)
ax.add_patch(grid_rect)

# Grid lines to show 0.01 deg resolution concept
for lon in np.arange(-125, -65, 5):
    ax.axvline(lon, color='#1e2a45', linewidth=0.3, alpha=0.5)
for lat in np.arange(22, 53, 5):
    ax.axhline(lat, color='#1e2a45', linewidth=0.3, alpha=0.5)

# US outline
ax.plot(us_outline[:, 0], us_outline[:, 1], color='#90a4c0', linewidth=1.5, zorder=3)
ax.fill(us_outline[:, 0], us_outline[:, 1], color='#1a2744', alpha=0.7, zorder=2)

# Scatter approximate NEXRAD radar locations (subset)
radar_lons = np.array([
    -122.5, -121.6, -118.2, -117.0, -112.0, -111.9, -106.6, -104.2,
    -100.5, -97.4, -95.7, -94.4, -93.6, -91.2, -89.8, -87.5, -86.3,
    -85.5, -84.3, -82.4, -81.0, -80.2, -78.5, -76.6, -75.0, -73.2,
    -71.5, -68.7, -98.1, -96.2, -92.6, -88.1, -85.0, -83.3, -77.0,
    -72.5, -115.0, -108.5, -103.5, -97.0, -91.0, -86.0, -80.0, -74.5,
    -119.6, -113.0, -105.0, -99.3, -93.8, -88.5, -82.0, -76.0, -70.5,
    -116.8, -110.0, -102.0, -96.0, -90.0, -84.0, -78.0, -72.0,
])
radar_lats = np.array([
    47.7, 45.5, 34.0, 32.9, 40.5, 33.5, 35.2, 39.8,
    31.4, 32.5, 29.7, 30.7, 32.5, 30.5, 33.0, 34.9, 33.0,
    30.4, 33.4, 28.0, 32.0, 25.6, 34.2, 36.8, 40.0, 41.4,
    42.0, 44.5, 38.0, 36.5, 35.0, 37.2, 35.2, 42.7, 38.5,
    44.0, 43.5, 42.0, 43.5, 40.5, 42.0, 42.5, 41.0, 42.5,
    48.0, 47.5, 47.0, 47.3, 47.0, 46.5, 46.0, 44.5, 46.0,
    38.0, 37.5, 37.0, 37.5, 37.0, 37.0, 37.0, 37.0,
])

# Draw radar coverage circles
for lon, lat in zip(radar_lons, radar_lats):
    circle = plt.Circle((lon, lat), 2.2, color=ACCENT2, alpha=0.06, zorder=4)
    ax.add_patch(circle)

ax.scatter(radar_lons, radar_lats, s=12, color=ACCENT2, alpha=0.8, zorder=5, label='NEXRAD radars (~180)')

# Annotations
ax.annotate('MRMS Grid Coverage\n3500 x 7000 @ 0.01\u00b0',
            xy=(-128, 53), fontsize=13, fontweight='bold', color=ACCENT,
            ha='left', va='top')
ax.annotate('24.5M grid points\n2-min update cadence',
            xy=(-128, 50.5), fontsize=10, color=DIM, ha='left', va='top')

ax.set_xlim(-133, -62)
ax.set_ylim(22, 52)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('MRMS: 180+ Radars Fused Into One Seamless Product', fontsize=16, fontweight='bold', pad=15)
ax.legend(loc='lower right', fontsize=10, framealpha=0.3, edgecolor='#2a3550')
ax.set_aspect(1.3)

plt.tight_layout()
plt.show()

---
## Slide 2: MRMS Data Deep Dive -- Tilt-Level Reflectivity

We fetch **tilt-level reflectivity** (`MergedReflectivityQC`) -- not the pre-computed 2D composite. Each timestamp gives us 8 elevation angles, which means:

- **One fetch path** serves both 2D (composite via `fmax`) and 3D (stacked tilt planes)
- Full CONUS grid per tilt: **3500 rows x 7000 cols** at 0.01° resolution
- 8 tilts per timestamp = **196 million data points** every 2 minutes
- Files are gzipped GRIB2 on S3: `s3://noaa-mrms-pds/CONUS/MergedReflectivityQC_{tilt}/{date}/...grib2.gz`

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

tilts_deg = [0.5, 1.0, 1.5, 2.5, 4.0, 7.0, 10.0, 19.0]
heights_km = [1.0, 1.5, 2.0, 3.5, 5.5, 9.0, 12.0, 19.0]
tilt_labels = ['00.50\u00b0', '01.00\u00b0', '01.50\u00b0', '02.50\u00b0',
               '04.00\u00b0', '07.00\u00b0', '10.00\u00b0', '19.00\u00b0']

colors = plt.cm.cool(np.linspace(0.15, 0.95, 8))

# Draw radar site
ax.plot(0, 0, 's', color=ACCENT2, markersize=14, zorder=10)
ax.annotate('Radar\nSite', xy=(0, 0), xytext=(-18, -1.5),
            fontsize=10, color=ACCENT2, ha='center', fontweight='bold')

# Draw earth surface
ax.axhline(0, color='#3a4a60', linewidth=2, linestyle='-', alpha=0.5)
ax.fill_between([-10, 260], [-2, -2], [0, 0], color='#1a2744', alpha=0.5)

max_range = 250  # km range

for i, (tilt, height, label) in enumerate(zip(tilts_deg, heights_km, tilt_labels)):
    # Draw beam line from radar to max range
    angle_rad = np.radians(tilt)
    x_end = max_range * np.cos(angle_rad)
    y_end = max_range * np.sin(angle_rad)

    # Beam with thickness
    beam_width = 0.45  # degrees
    upper = np.radians(tilt + beam_width)
    lower = np.radians(tilt - beam_width)

    xs = np.linspace(0, max_range, 100)
    y_upper = xs * np.sin(upper)
    y_lower = xs * np.sin(lower)
    x_pts = xs * np.cos(np.radians(tilt))

    ax.fill_between(x_pts, y_lower, y_upper, color=colors[i], alpha=0.15)
    ax.plot(x_pts, xs * np.sin(angle_rad), color=colors[i], linewidth=1.5, alpha=0.8)

    # Label at end of beam
    ax.annotate(f'{label}\n{height:.0f} km',
                xy=(x_end + 3, y_end),
                fontsize=9, color=colors[i], fontweight='bold',
                va='center', ha='left')

# Height markers on right side
for h in [5, 10, 15, 20]:
    ax.axhline(h, color='#2a3550', linewidth=0.5, linestyle=':', alpha=0.5)
    ax.text(-8, h, f'{h} km', fontsize=8, color=DIM, va='center', ha='right')

ax.set_xlim(-20, 290)
ax.set_ylim(-2.5, 25)
ax.set_xlabel('Range from Radar (km)', fontsize=12)
ax.set_ylabel('Height (km)', fontsize=12)
ax.set_title('8 Tilt Elevation Angles: From Ground Level to 19 km', fontsize=16, fontweight='bold', pad=15)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---
## Slide 3: Why MRMS Excels for Weather Forecasting

| Feature | MRMS | Traditional Single Radar |
|---|---|---|
| **Update cadence** | 2 minutes | 4-6 minutes |
| **Coverage** | All CONUS (seamless) | ~230 km radius (gaps between sites) |
| **Resolution** | 0.01° (~1 km) | Variable (0.5-1 km near site, degrades with range) |
| **Vertical levels** | 33 tilts available, we use 8 | Depends on VCP scan strategy |
| **Access** | Free S3 bucket, no auth | Requires Level-II archive access |
| **Processing** | Pre-fused, quality-controlled | Raw sweeps need dealiasing, clutter removal |

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Timeline: 20 minutes of data
minutes = np.arange(0, 21, 1)

# MRMS: every 2 minutes
mrms_times = np.arange(0, 21, 2)
# Traditional: every 5 minutes
trad_times = np.arange(0, 21, 5)

y_mrms = 2
y_trad = 0.8

# Background timeline
ax.axhline(y_mrms, color='#2a3550', linewidth=1, xmin=0.05, xmax=0.95)
ax.axhline(y_trad, color='#2a3550', linewidth=1, xmin=0.05, xmax=0.95)

# MRMS updates
for t in mrms_times:
    ax.plot(t, y_mrms, 'o', color=ACCENT, markersize=14, zorder=5)
    ax.plot([t, t], [y_mrms - 0.15, y_mrms + 0.15], color=ACCENT, linewidth=2)

# Traditional updates
for t in trad_times:
    ax.plot(t, y_trad, 's', color=ACCENT3, markersize=14, zorder=5)
    ax.plot([t, t], [y_trad - 0.15, y_trad + 0.15], color=ACCENT3, linewidth=2)

# Labels
ax.text(-1.5, y_mrms, 'MRMS\n(2-min)', fontsize=12, fontweight='bold', color=ACCENT,
        va='center', ha='right')
ax.text(-1.5, y_trad, 'Traditional\n(5-min)', fontsize=12, fontweight='bold', color=ACCENT3,
        va='center', ha='right')

# Count annotations
ax.annotate(f'{len(mrms_times)} frames in 20 min', xy=(21.5, y_mrms),
            fontsize=11, color=ACCENT, va='center', fontweight='bold')
ax.annotate(f'{len(trad_times)} frames in 20 min', xy=(21.5, y_trad),
            fontsize=11, color=ACCENT3, va='center', fontweight='bold')

# Time axis
for t in range(0, 21, 2):
    ax.text(t, 0.15, f'{t}m', fontsize=8, color=DIM, ha='center')

ax.set_xlim(-5, 28)
ax.set_ylim(-0.2, 2.8)
ax.set_title('Update Cadence: 2.5x More Temporal Resolution', fontsize=16, fontweight='bold', pad=15)
ax.axis('off')

# Key insight box
bbox_props = dict(boxstyle='round,pad=0.5', facecolor='#1a2744', edgecolor=ACCENT, alpha=0.9)
ax.text(10, -0.05, 'Higher cadence = smoother animation + earlier storm detection',
        fontsize=11, color=ACCENT, ha='center', va='center', bbox=bbox_props)

plt.tight_layout()
plt.show()

---
## Slide 4: The Sparsity Problem (and Our Solution)

Weather radar data has a fundamental property that dominates the engineering:

**95-99% of the CONUS grid is empty at any given time.** Most of the continent isn't experiencing precipitation.

A single timestamp (8 tilts) as dense float32: **784 MB**. Caching 30 frames for smooth animation would require **23.5 GB** -- impossible.

**Our solution:** `scipy.sparse.csr_matrix` -- Compressed Sparse Row format. Only store non-zero (precipitating) grid cells.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={'width_ratios': [1, 1.3]})

# Left: Sparsity visualization
ax = axes[0]
np.random.seed(42)
grid_size = 50
grid = np.zeros((grid_size, grid_size))

# Simulate sparse precipitation (~3% coverage)
n_storms = 4
for _ in range(n_storms):
    cx, cy = np.random.randint(5, 45, 2)
    radius = np.random.randint(2, 6)
    for dx in range(-radius, radius + 1):
        for dy in range(-radius, radius + 1):
            if dx*dx + dy*dy <= radius*radius:
                nx, ny = cx + dx, cy + dy
                if 0 <= nx < grid_size and 0 <= ny < grid_size:
                    grid[nx, ny] = np.random.uniform(10, 60)

# NWS-like colormap
from matplotlib.colors import LinearSegmentedColormap
nws_colors = [
    (0.0, '#0f1729'),
    (0.01, '#0f1729'),
    (0.08, '#40c040'),
    (0.25, '#30a030'),
    (0.4, '#ffff00'),
    (0.55, '#ff6400'),
    (0.7, '#ff0000'),
    (0.85, '#b400c8'),
    (1.0, '#ffffff'),
]
nws_cmap = LinearSegmentedColormap.from_list('nws', nws_colors)

im = ax.imshow(grid, cmap=nws_cmap, vmin=0, vmax=65, aspect='equal', interpolation='nearest')

total_cells = grid_size * grid_size
active_cells = np.count_nonzero(grid)
pct_empty = (1 - active_cells / total_cells) * 100

ax.set_title(f'Typical MRMS Frame: {pct_empty:.0f}% Empty', fontsize=14, fontweight='bold', pad=10)
ax.set_xlabel(f'{active_cells} of {total_cells} cells active', fontsize=10, color=DIM)
ax.tick_params(labelbottom=False, labelleft=False)

# Right: Bar chart comparison
ax2 = axes[1]

categories = ['Memory\n(per timestamp)', 'Disk Size\n(per timestamp)', 'Load Time\n(from disk)', 'LRU Capacity\n(in 1.2 GB)']
dense_vals = [784, 7.1, 295, 1.5]
sparse_vals = [39, 5.4, 33, 30]
units = ['MB', 'MB', 'ms', 'frames']

x = np.arange(len(categories))
width = 0.35

bars1 = ax2.bar(x - width/2, dense_vals, width, color=WARN, alpha=0.8, label='Dense (float32)')
bars2 = ax2.bar(x + width/2, sparse_vals, width, color=ACCENT2, alpha=0.8, label='Sparse (CSR)')

# Value labels on bars
for bar, val, unit in zip(bars1, dense_vals, units):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
             f'{val} {unit}', ha='center', va='bottom', fontsize=9, color=WARN)
for bar, val, unit in zip(bars2, sparse_vals, units):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
             f'{val} {unit}', ha='center', va='bottom', fontsize=9, color=ACCENT2)

# Improvement annotations
improvements = ['20x', '1.3x', '9x', '20x']
for i, imp in enumerate(improvements):
    ax2.annotate(f'{imp} better', xy=(i, max(dense_vals[i], sparse_vals[i]) + 40),
                fontsize=10, fontweight='bold', color=ACCENT, ha='center')

ax2.set_xticks(x)
ax2.set_xticklabels(categories, fontsize=10)
ax2.set_ylabel('Value', fontsize=12)
ax2.set_title('Dense vs Sparse: Every Metric Improves', fontsize=14, fontweight='bold', pad=10)
ax2.legend(fontsize=11, loc='upper right', framealpha=0.3, edgecolor='#2a3550')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.set_ylim(0, max(dense_vals) * 1.2)

plt.tight_layout()
plt.show()

---
## Slide 5: System Architecture

The system has three layers:

1. **NOAA S3 Bucket** (`noaa-mrms-pds`) -- public, free, no credentials. Files appear with ~2-3 min latency.
2. **Python Backend** (FastAPI) -- fetches, decodes, caches sparse tilt grids, renders atlas tiles on demand, computes motion fields via FFT.
3. **Browser Frontend** (MapLibre + three.js) -- GPU-side dBZ decoding, NWS color ramp, motion-compensated advection, volume rendering. All in GLSL shaders.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 9))
ax.set_xlim(0, 16)
ax.set_ylim(0, 9)
ax.axis('off')

def draw_box(ax, x, y, w, h, label, sublabel='', color=ACCENT, fontsize=13):
    box = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.15',
                         facecolor=color, alpha=0.15, edgecolor=color, linewidth=2)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2 + (0.15 if sublabel else 0), label,
            ha='center', va='center', fontsize=fontsize, fontweight='bold', color=color)
    if sublabel:
        ax.text(x + w/2, y + h/2 - 0.25, sublabel,
                ha='center', va='center', fontsize=9, color=DIM)

def draw_arrow(ax, x1, y1, x2, y2, label='', color='#5a6680'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))
    if label:
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        ax.text(mx, my + 0.25, label, ha='center', va='bottom', fontsize=9, color=color)

# NOAA S3
draw_box(ax, 0.5, 5.5, 3, 1.8, 'NOAA S3 Bucket', 'noaa-mrms-pds (public)', '#4fc3f7')

# Backend components
draw_box(ax, 5, 7, 3, 1.2, 'GRIB2 Decoder', 'Pure Python, no eccodes', ACCENT2)
draw_box(ax, 5, 5.3, 3, 1.2, 'Sparse Cache', 'scipy CSR + LRU (30 frames)', ACCENT2)
draw_box(ax, 5, 3.6, 3, 1.2, 'Atlas Tile Renderer', '256x2048 PNG per tile', ACCENT2)
draw_box(ax, 5, 1.9, 3, 1.2, 'Motion Engine', 'FFT block matching', ACCENT2)

# Backend group label
ax.text(6.5, 8.6, 'Python Backend (FastAPI)', fontsize=14, fontweight='bold',
        color=ACCENT2, ha='center')
rect = mpatches.FancyBboxPatch((4.7, 1.6), 3.6, 7.2, boxstyle='round,pad=0.2',
                                facecolor='none', edgecolor=ACCENT2, linewidth=1, linestyle='--', alpha=0.3)
ax.add_patch(rect)

# Frontend components
draw_box(ax, 10, 7, 3, 1.2, 'MapLibre GL JS', 'Base map + projection', ACCENT3)
draw_box(ax, 10, 5.3, 3, 1.2, 'three.js + GLSL', 'Radar overlay shaders', ACCENT3)
draw_box(ax, 10, 3.6, 3, 1.2, '3 View Modes', 'Composite / 3D / Volume', ACCENT3)
draw_box(ax, 10, 1.9, 3, 1.2, 'Motion Advection', 'Semi-Lagrangian in shader', ACCENT3)

# Frontend group label
ax.text(11.5, 8.6, 'Browser Frontend (WebGL)', fontsize=14, fontweight='bold',
        color=ACCENT3, ha='center')
rect2 = mpatches.FancyBboxPatch((9.7, 1.6), 3.6, 7.2, boxstyle='round,pad=0.2',
                                 facecolor='none', edgecolor=ACCENT3, linewidth=1, linestyle='--', alpha=0.3)
ax.add_patch(rect2)

# User
draw_box(ax, 14, 4.8, 1.5, 2.2, 'User', 'Browser', '#b39ddb', fontsize=14)

# Arrows
draw_arrow(ax, 3.5, 6.4, 5, 7.6, '.grib2.gz', ACCENT)
draw_arrow(ax, 6.5, 7.0, 6.5, 6.5, '', DIM)
draw_arrow(ax, 6.5, 5.3, 6.5, 4.8, '', DIM)
draw_arrow(ax, 6.5, 3.6, 6.5, 3.1, '', DIM)

draw_arrow(ax, 8, 4.2, 10, 5.9, 'Atlas PNGs', ACCENT2)
draw_arrow(ax, 8, 2.5, 10, 2.5, 'Motion PNGs', ACCENT2)

draw_arrow(ax, 13, 5.9, 14, 5.9, '', ACCENT3)

# Title
ax.set_title('System Architecture: Three-Layer Pipeline', fontsize=18, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

---
## Slide 6: The Data Pipeline

### Startup Seed (60 frames, ~1-2 minutes)
1. List 60 most recent `00.50` tilt keys from S3
2. For each timestamp, derive keys for all 8 tilts
3. Fetch each tilt in parallel (ThreadPoolExecutor, 8 workers)
4. Decompress `.grib2.gz`, decode with our custom GRIB2 decoder
5. Mask sentinels (`data[data < -30] = NaN`), convert to `scipy.sparse.csr_matrix`
6. Save to disk (`data/tilt_grids/{timestamp}/`), populate in-memory LRU
7. Pre-render atlas tiles for default CONUS viewport
8. Compute motion fields for all consecutive frame pairs

### Periodic Refresh (every 60 seconds)
- Check S3 for new frames, fetch only new ones
- Compute motion fields for new pairs
- Purge entries older than 3 hours, cap at 60 frames

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.set_xlim(0, 16)
ax.set_ylim(0, 6)
ax.axis('off')

stages = [
    ('S3 Listing', '60 timestamps\n3 days scanned', 0.3, ACCENT, '~2s'),
    ('Parallel\nFetch', '480 files\n8 workers', 2.8, ACCENT, '~60s'),
    ('GRIB2\nDecode', 'struct + numpy\n+ Pillow J2K', 5.3, ACCENT2, '~5ms/file'),
    ('Sparse\nConvert', 'CSR matrices\n20x savings', 7.8, ACCENT2, '~10ms/file'),
    ('Disk +\nLRU Cache', '.npz + meta.json\n30 entries', 10.3, '#b39ddb', '~1.2s/ts'),
    ('Atlas Tiles\n+ Motion', 'PNG rendering\nFFT matching', 12.8, ACCENT3, '~0.5s/pair'),
]

y_center = 3.5
box_w, box_h = 2.0, 2.0

for i, (name, detail, x, color, timing) in enumerate(stages):
    # Stage box
    box = FancyBboxPatch((x, y_center - box_h/2), box_w, box_h,
                         boxstyle='round,pad=0.15',
                         facecolor=color, alpha=0.12, edgecolor=color, linewidth=2)
    ax.add_patch(box)

    # Stage number
    ax.text(x + 0.2, y_center + box_h/2 - 0.25, str(i + 1),
            fontsize=18, fontweight='bold', color=color, alpha=0.4)

    # Stage name
    ax.text(x + box_w/2, y_center + 0.25, name,
            ha='center', va='center', fontsize=11, fontweight='bold', color=color)

    # Detail
    ax.text(x + box_w/2, y_center - 0.4, detail,
            ha='center', va='center', fontsize=8, color=DIM)

    # Timing below
    ax.text(x + box_w/2, y_center - box_h/2 - 0.3, timing,
            ha='center', va='center', fontsize=9, color=color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.1, edgecolor='none'))

    # Arrow to next stage
    if i < len(stages) - 1:
        next_x = stages[i + 1][2]
        ax.annotate('', xy=(next_x, y_center), xytext=(x + box_w, y_center),
                    arrowprops=dict(arrowstyle='->', color=DIM, lw=1.5))

ax.set_title('Data Pipeline: From S3 to Cached Atlas Tiles', fontsize=16, fontweight='bold', pad=15)

# Total timing annotation
bbox_props = dict(boxstyle='round,pad=0.4', facecolor='#1a2744', edgecolor=ACCENT, alpha=0.9)
ax.text(8, 0.8, 'Full seed: ~90 seconds for 60 frames (2 hours of radar history)',
        fontsize=11, color=ACCENT, ha='center', va='center', bbox=bbox_props)

plt.tight_layout()
plt.show()

---
## Slide 7: Custom GRIB2 Decoder

GRIB2 is the WMO standard binary format for meteorological data. The standard library (eccodes) is a **100+ MB C library** with complex cross-platform installation. We don't need any of that.

MRMS reflectivity uses a tiny subset of the GRIB2 spec:
- **1 grid template** (3.0: regular lat/lon)
- **3 packing templates** (5.0: simple, 5.40: JPEG2000, 5.41: PNG)
- **1 product family** (reflectivity)

Our decoder: ~400 lines of Python using `struct` + `numpy` + `Pillow` (for JPEG2000). Zero C dependencies.

**Gotchas we hit:**
- GRIB2 uses explicit sign-bit encoding (MSB), NOT two's-complement
- Longitude encoding uses unsigned wrapping, not the sign-bit convention
- Section 3 byte offsets don't match naive reading of the 1-based GRIB2 spec

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: eccodes dependency tree
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_title('Traditional: eccodes Stack', fontsize=14, fontweight='bold', color=WARN, pad=10)

eccodes_deps = [
    ('eccodes (C library)', 5, 9, WARN, 13),
    ('libeccodes.so\n~100 MB', 2.5, 7.2, WARN, 10),
    ('cfgrib / pygrib', 7.5, 7.2, WARN, 10),
    ('cmake', 1, 5.5, '#ef9a9a', 9),
    ('libjasper', 3.5, 5.5, '#ef9a9a', 9),
    ('libpng-dev', 5.5, 5.5, '#ef9a9a', 9),
    ('libaec', 7.5, 5.5, '#ef9a9a', 9),
    ('xarray', 5, 3.8, '#ef9a9a', 9),
    ('Platform-specific\nbuild toolchains', 5, 2, '#ef9a9a', 10),
]

for label, x, y, color, fs in eccodes_deps:
    box = FancyBboxPatch((x - 1.2, y - 0.5), 2.4, 1,
                         boxstyle='round,pad=0.1',
                         facecolor=color, alpha=0.12, edgecolor=color, linewidth=1.5)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center', fontsize=fs, color=color)

# Connection lines
for x2, y2 in [(2.5, 7.7), (7.5, 7.7)]:
    ax.plot([5, x2], [8.5, y2], color=WARN, alpha=0.3, linewidth=1)
for x2 in [1, 3.5, 5.5, 7.5]:
    ax.plot([5, x2], [6.7, 6.0], color='#ef9a9a', alpha=0.3, linewidth=1)

# Big X
ax.text(5, 1, 'X', fontsize=60, color=WARN, alpha=0.15, ha='center', va='center', fontweight='bold')

# Right: Our minimal decoder
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('Our Approach: Minimal Pure Python', fontsize=14, fontweight='bold', color=ACCENT2, pad=10)

our_deps = [
    ('grib2/ decoder', 5, 8.5, ACCENT2, 14),
    ('struct\n(stdlib)', 2, 6.5, ACCENT, 11),
    ('numpy', 5, 6.5, ACCENT, 11),
    ('Pillow\n(for J2K)', 8, 6.5, ACCENT, 11),
]

for label, x, y, color, fs in our_deps:
    box = FancyBboxPatch((x - 1.2, y - 0.55), 2.4, 1.1,
                         boxstyle='round,pad=0.1',
                         facecolor=color, alpha=0.15, edgecolor=color, linewidth=2)
    ax2.add_patch(box)
    ax2.text(x, y, label, ha='center', va='center', fontsize=fs, color=color, fontweight='bold')

for x2 in [2, 5, 8]:
    ax2.plot([5, x2], [7.95, 7.05], color=ACCENT2, alpha=0.4, linewidth=1.5)

# Stats
stats = [
    ('~400 lines of Python', 5, 4.8, 14),
    ('0 C dependencies', 5, 3.8, 14),
    ('3 packing templates', 5, 2.8, 14),
    ('pip install: 5 seconds', 5, 1.8, 14),
]

for label, x, y, fs in stats:
    ax2.text(x, y, label, ha='center', va='center', fontsize=fs, color=ACCENT2, fontweight='bold')
    ax2.text(x - 2.5, y, '\u2713', ha='center', va='center', fontsize=18, color=ACCENT2)

plt.tight_layout()
plt.show()

---
## Slide 8: Atlas Tiles + GPU Rendering

The frontend doesn't get raw data -- it gets **atlas tiles**: pre-rendered grayscale PNGs where all the intelligence is in the GLSL shaders.

### Atlas Tile Format (256 x 2048 grayscale PNG)
- 8 vertical bands of 256 x 256, one per tilt level
- dBZ encoded as `uint8 = round((dBZ + 30) * 2)`, 0 = no echo
- Typically 2-10 KB per tile (extremely compressible)

### GPU-Side Processing (GLSL Shaders)
- **dBZ decode:** `dBZ = pixel / 2.0 - 30.0`
- **NWS color ramp:** texture lookup from a 256-wide color ramp
- **Compositing:** `fmax` across 8 bands (composite mode)
- **Motion advection:** semi-Lagrangian backward/forward trace
- **Volume rendering:** ray marching with trilinear interpolation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7), gridspec_kw={'width_ratios': [0.4, 1]})

# Left: Atlas tile format visualization
ax = axes[0]
ax.set_xlim(0, 4)
ax.set_ylim(-0.5, 17)
ax.axis('off')
ax.set_title('Atlas Tile Format\n(256 x 2048 PNG)', fontsize=13, fontweight='bold', pad=10)

tilt_labels = ['00.50\u00b0 (1 km)', '01.00\u00b0 (1.5 km)', '01.50\u00b0 (2 km)',
               '02.50\u00b0 (3.5 km)', '04.00\u00b0 (5.5 km)', '07.00\u00b0 (9 km)',
               '10.00\u00b0 (12 km)', '19.00\u00b0 (19 km)']
colors_tilt = plt.cm.cool(np.linspace(0.15, 0.95, 8))

for i in range(8):
    y = 15 - i * 2
    rect = FancyBboxPatch((0.5, y), 2, 1.6, boxstyle='round,pad=0.05',
                          facecolor=colors_tilt[i], alpha=0.2, edgecolor=colors_tilt[i], linewidth=1.5)
    ax.add_patch(rect)
    ax.text(1.5, y + 0.8, f'Band {i}', ha='center', va='center',
            fontsize=9, fontweight='bold', color=colors_tilt[i])
    ax.text(3.2, y + 0.8, tilt_labels[i], ha='left', va='center',
            fontsize=8, color=DIM)

# Right: NWS Reflectivity Color Scale
ax2 = axes[1]

nws_bands = [
    (5, 10, (64, 192, 64), 'Light Green'),
    (10, 15, (48, 160, 48), ''),
    (15, 20, (0, 144, 0), 'Green'),
    (20, 25, (0, 120, 0), ''),
    (25, 30, (255, 255, 0), 'Yellow'),
    (30, 35, (230, 180, 0), 'Gold'),
    (35, 40, (255, 100, 0), 'Orange'),
    (40, 45, (255, 0, 0), 'Red'),
    (45, 50, (200, 0, 0), 'Dark Red'),
    (50, 55, (180, 0, 120), 'Magenta'),
    (55, 60, (150, 0, 200), 'Purple'),
    (60, 65, (255, 255, 255), 'White'),
    (65, 75, (200, 200, 255), 'Light Blue'),
]

# Severity labels
severity_ranges = [
    (5, 20, 'Light rain / drizzle', ACCENT2),
    (20, 35, 'Moderate rain', ACCENT3),
    (35, 50, 'Heavy rain / hail risk', WARN),
    (50, 75, 'Severe / destructive', '#e040fb'),
]

for lo, hi, rgb, name in nws_bands:
    color = tuple(c/255 for c in rgb)
    ax2.barh(lo, hi - lo, left=0, height=4.5, color=color, alpha=0.85)
    label_color = '#0f1729' if sum(rgb) > 400 else '#e0e6f0'
    if name:
        ax2.text((hi - lo) / 2, lo + 2.25, name, ha='center', va='center',
                fontsize=8, color=label_color, fontweight='bold')

# dBZ labels
for lo, hi, _, _ in nws_bands:
    ax2.text(-1, lo + 2.25, f'{lo}', ha='right', va='center', fontsize=9, color=DIM)

# Severity annotations
for lo, hi, label, color in severity_ranges:
    mid = (lo + hi) / 2
    ax2.annotate(label, xy=(12, mid), fontsize=10, color=color, fontweight='bold',
                va='center', ha='left')
    ax2.plot([10.5, 11.5], [mid, mid], color=color, linewidth=2)

ax2.set_xlim(-2, 28)
ax2.set_ylim(2, 78)
ax2.set_title('NWS Reflectivity Color Scale (dBZ)', fontsize=14, fontweight='bold', pad=10)
ax2.set_ylabel('Reflectivity (dBZ)', fontsize=11)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['bottom'].set_visible(False)
ax2.tick_params(labelbottom=False, bottom=False)

plt.tight_layout()
plt.show()

---
## Slide 9: Motion-Compensated Interpolation

MRMS updates every 2 minutes. A naive crossfade between frames causes storms to **ghost/pulse** -- sharp at keyframes, blurry in between. Our solution: **slide storm cells along their displacement vectors.**

### Backend: FFT Block Matching
1. Build 2D composite via `fmax` across 8 tilts
2. Downsample 8x for efficiency (~438 x 875)
3. For each 32x32 block (stride 16), find best match in next frame via `scipy.signal.fftconvolve`
4. Record displacement (U, V) in degrees + NCC confidence
5. Encode as RGB PNG: R=U, G=V, B=confidence (~2-5 KB per frame pair)

### Frontend: Semi-Lagrangian Advection (GLSL)
- Trace **backward** by `t * displacement` to sample frame A
- Trace **forward** by `(1-t) * displacement` to sample frame B
- Blend: `mix(crossfade, advected, confidence)` -- falls back gracefully where motion is unreliable

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

def draw_storm(ax, cx, cy, radius=8, intensity=1.0, label=None):
    """Draw a simplified storm cell."""
    grid = np.zeros((50, 50))
    Y, X = np.mgrid[0:50, 0:50]
    dist = np.sqrt((X - cx)**2 + (Y - cy)**2)
    mask = dist < radius
    grid[mask] = intensity * np.clip(1 - dist[mask] / radius, 0, 1) * 60
    return grid

# Storm positions at Frame A and Frame B
storm_a_x, storm_a_y = 15, 25
storm_b_x, storm_b_y = 30, 25

frame_a = draw_storm(np.zeros(1), storm_a_x, storm_a_y)
frame_b = draw_storm(np.zeros(1), storm_b_x, storm_b_y)

# Create frames
grid_a = np.zeros((50, 50))
grid_b = np.zeros((50, 50))
Y, X = np.mgrid[0:50, 0:50]

dist_a = np.sqrt((X - storm_a_x)**2 + (Y - storm_a_y)**2)
dist_b = np.sqrt((X - storm_b_x)**2 + (Y - storm_b_y)**2)
r = 8
grid_a[dist_a < r] = np.clip(1 - dist_a[dist_a < r] / r, 0, 1) * 60
grid_b[dist_b < r] = np.clip(1 - dist_b[dist_b < r] / r, 0, 1) * 60

from matplotlib.colors import LinearSegmentedColormap
nws_colors = [
    (0.0, '#0f1729'), (0.01, '#0f1729'), (0.08, '#40c040'),
    (0.25, '#30a030'), (0.4, '#ffff00'), (0.55, '#ff6400'),
    (0.7, '#ff0000'), (0.85, '#b400c8'), (1.0, '#ffffff'),
]
nws_cmap = LinearSegmentedColormap.from_list('nws', nws_colors)

# Top row: Crossfade (naive)
t = 0.5
crossfade = grid_a * (1 - t) + grid_b * t

axes[0, 0].imshow(grid_a, cmap=nws_cmap, vmin=0, vmax=65, aspect='equal')
axes[0, 0].set_title('Frame A (t=0)', fontsize=12, fontweight='bold', color=ACCENT)
axes[0, 0].tick_params(labelbottom=False, labelleft=False)

axes[0, 1].imshow(crossfade, cmap=nws_cmap, vmin=0, vmax=65, aspect='equal')
axes[0, 1].set_title('Naive Crossfade (t=0.5)', fontsize=12, fontweight='bold', color=WARN)
axes[0, 1].tick_params(labelbottom=False, labelleft=False)
axes[0, 1].annotate('Ghosting!\nTwo blurry copies', xy=(22, 40), fontsize=10, color=WARN,
                    fontweight='bold', ha='center',
                    bbox=dict(boxstyle='round', facecolor='#0f1729', alpha=0.8, edgecolor=WARN))

axes[0, 2].imshow(grid_b, cmap=nws_cmap, vmin=0, vmax=65, aspect='equal')
axes[0, 2].set_title('Frame B (t=1)', fontsize=12, fontweight='bold', color=ACCENT)
axes[0, 2].tick_params(labelbottom=False, labelleft=False)

# Bottom row: Motion-compensated
# Simulate advected interpolation (storm at midpoint, sharp)
mid_x = storm_a_x + (storm_b_x - storm_a_x) * t
mid_y = storm_a_y + (storm_b_y - storm_a_y) * t
dist_mid = np.sqrt((X - mid_x)**2 + (Y - mid_y)**2)
grid_advected = np.zeros((50, 50))
grid_advected[dist_mid < r] = np.clip(1 - dist_mid[dist_mid < r] / r, 0, 1) * 60

# Show motion vectors
axes[1, 0].imshow(grid_a, cmap=nws_cmap, vmin=0, vmax=65, aspect='equal')
axes[1, 0].annotate('', xy=(storm_b_x, storm_b_y), xytext=(storm_a_x, storm_a_y),
                    arrowprops=dict(arrowstyle='->', color=ACCENT3, lw=3))
axes[1, 0].set_title('Motion Vectors (FFT)', fontsize=12, fontweight='bold', color=ACCENT3)
axes[1, 0].tick_params(labelbottom=False, labelleft=False)
axes[1, 0].text(22, 22, 'displacement', fontsize=9, color=ACCENT3, ha='center')

axes[1, 1].imshow(grid_advected, cmap=nws_cmap, vmin=0, vmax=65, aspect='equal')
axes[1, 1].set_title('Motion-Compensated (t=0.5)', fontsize=12, fontweight='bold', color=ACCENT2)
axes[1, 1].tick_params(labelbottom=False, labelleft=False)
axes[1, 1].annotate('Sharp!\nOne crisp cell', xy=(22, 40), fontsize=10, color=ACCENT2,
                    fontweight='bold', ha='center',
                    bbox=dict(boxstyle='round', facecolor='#0f1729', alpha=0.8, edgecolor=ACCENT2))

# Formula / process
axes[1, 2].set_xlim(0, 10)
axes[1, 2].set_ylim(0, 10)
axes[1, 2].axis('off')
axes[1, 2].set_title('Semi-Lagrangian Advection', fontsize=12, fontweight='bold', color=ACCENT)

steps = [
    (5, 9, 'For each pixel at time t:', ACCENT, 11),
    (5, 7.5, '1. Trace BACK by t*disp', ACCENT, 11),
    (5, 6.5, '   \u2192 sample Frame A', DIM, 10),
    (5, 5, '2. Trace FORWARD by (1-t)*disp', ACCENT, 11),
    (5, 4, '   \u2192 sample Frame B', DIM, 10),
    (5, 2.5, '3. Blend by confidence:', ACCENT, 11),
    (5, 1.3, 'mix(crossfade, advected, conf)', ACCENT2, 11),
]
for x, y, text, color, fs in steps:
    axes[1, 2].text(x, y, text, ha='center', va='center', fontsize=fs,
                   color=color, fontweight='bold' if color != DIM else 'normal',
                   family='monospace' if 'mix(' in text else 'sans-serif')

# Row labels
fig.text(0.02, 0.73, 'NAIVE', fontsize=14, fontweight='bold', color=WARN,
         rotation=90, va='center')
fig.text(0.02, 0.3, 'OURS', fontsize=14, fontweight='bold', color=ACCENT2,
         rotation=90, va='center')

plt.suptitle('Crossfade vs Motion-Compensated Interpolation', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Slide 10: Three View Modes from One Data Source

The same atlas tile data (8 tilt bands per tile) drives three completely different visualizations, all in GLSL:

| Mode | Technique | What you see |
|---|---|---|
| **Composite** | `fmax` across 8 bands on one ground plane | Classic 2D radar view |
| **3D** | 8 stacked translucent planes at tilt altitudes | Storm structure at each elevation |
| **Volume** | Ray-marched box with trilinear interpolation | Continuous 3D storm volume |

The frontend is **~37 KB minified** (vanilla JS, no framework). MapLibre and three.js loaded from CDN.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 6))

np.random.seed(123)

# Simulate storm data
grid_size = 40
Y, X = np.mgrid[0:grid_size, 0:grid_size]

def make_storm(cx, cy, rx, ry, peak):
    dist = ((X - cx)/rx)**2 + ((Y - cy)/ry)**2
    storm = np.zeros_like(dist)
    mask = dist < 1
    storm[mask] = peak * (1 - dist[mask])**1.5
    return storm

storms = (make_storm(15, 20, 8, 7, 55) + make_storm(28, 12, 5, 5, 45)
          + make_storm(10, 32, 6, 4, 35))

from matplotlib.colors import LinearSegmentedColormap
nws_colors = [
    (0.0, '#0f1729'), (0.02, '#0f1729'), (0.08, '#40c040'),
    (0.25, '#30a030'), (0.4, '#ffff00'), (0.55, '#ff6400'),
    (0.7, '#ff0000'), (0.85, '#b400c8'), (1.0, '#ffffff'),
]
nws_cmap = LinearSegmentedColormap.from_list('nws', nws_colors)

# Composite mode
ax = axes[0]
ax.imshow(storms, cmap=nws_cmap, vmin=0, vmax=65, aspect='equal', interpolation='bilinear')
ax.set_title('Composite Mode', fontsize=14, fontweight='bold', color=ACCENT)
ax.text(20, 37, 'fmax across 8 bands', fontsize=9, color=ACCENT, ha='center',
        bbox=dict(boxstyle='round', facecolor='#0f1729', alpha=0.85, edgecolor=ACCENT))
ax.tick_params(labelbottom=False, labelleft=False)

# 3D mode (stacked layers)
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')
ax2.set_title('3D Mode', fontsize=14, fontweight='bold', color=ACCENT3)

heights_km = [1.0, 1.5, 2.0, 3.5, 5.5, 9.0, 12.0, 19.0]
colors_3d = plt.cm.cool(np.linspace(0.15, 0.95, 8))

for i in range(8):
    h_norm = heights_km[i] / 20
    y = 1 + h_norm * 7
    # Perspective parallelogram
    skew = h_norm * 1.5
    xs = [1 + skew, 8 + skew, 9 + skew * 0.5, 2 + skew * 0.5]
    ys = [y, y, y + 0.6, y + 0.6]
    ax2.fill(xs, ys, color=colors_3d[i], alpha=0.25)
    ax2.plot(xs + [xs[0]], ys + [ys[0]], color=colors_3d[i], linewidth=1, alpha=0.6)

    # Small simulated radar blobs
    n_blobs = max(1, 4 - i // 2)
    for _ in range(n_blobs):
        bx = np.random.uniform(xs[0] + 0.5, xs[1] - 0.5)
        by = y + 0.3
        ax2.plot(bx, by, 'o', color=colors_3d[i], markersize=np.random.uniform(3, 8),
                alpha=0.5)

    ax2.text(9.5 + skew * 0.3, y + 0.3, f'{heights_km[i]:.0f}km',
            fontsize=7, color=colors_3d[i], va='center')

ax2.text(5, 0.3, '8 stacked planes', fontsize=9, color=ACCENT3, ha='center',
        bbox=dict(boxstyle='round', facecolor='#0f1729', alpha=0.85, edgecolor=ACCENT3))

# Volume mode
ax3 = axes[2]
ax3.set_xlim(0, 10)
ax3.set_ylim(0, 10)
ax3.axis('off')
ax3.set_title('Volume Mode', fontsize=14, fontweight='bold', color='#b39ddb')

# Draw a box with rays
box_xs = [2, 8, 9, 3, 2]
box_ys = [2, 2, 3, 3, 2]
box_top_xs = [3, 9, 9.5, 3.5, 3]
box_top_ys = [7, 7, 7.5, 7.5, 7]

# Box edges
ax3.plot(box_xs, box_ys, color='#b39ddb', linewidth=1.5, alpha=0.5)
ax3.plot(box_top_xs, box_top_ys, color='#b39ddb', linewidth=1.5, alpha=0.5)
for bx, by, tx, ty in zip(box_xs[:4], box_ys[:4], box_top_xs[:4], box_top_ys[:4]):
    ax3.plot([bx, tx], [by, ty], color='#b39ddb', linewidth=1, alpha=0.3)

# Volume blobs inside
for _ in range(15):
    bx = np.random.uniform(3.5, 8)
    by = np.random.uniform(3, 6.5)
    s = np.random.uniform(200, 800)
    c = nws_cmap(np.random.uniform(0.15, 0.8))
    ax3.scatter([bx], [by], s=s, color=c, alpha=0.15, edgecolors='none')

# Rays
eye_x, eye_y = 1, 9
for i in range(6):
    tx = 3 + i * 1.2
    ty = np.random.uniform(3, 6)
    ax3.annotate('', xy=(tx, ty), xytext=(eye_x, eye_y),
                arrowprops=dict(arrowstyle='->', color='#b39ddb', lw=0.8, alpha=0.4))

ax3.plot(eye_x, eye_y, 'o', color='#b39ddb', markersize=8)
ax3.text(eye_x, eye_y + 0.5, 'camera', fontsize=8, color='#b39ddb', ha='center')

ax3.text(5.5, 1, 'ray marching + trilinear', fontsize=9, color='#b39ddb', ha='center',
        bbox=dict(boxstyle='round', facecolor='#0f1729', alpha=0.85, edgecolor='#b39ddb'))

plt.suptitle('Three View Modes, One Data Source', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## Slide 11: Key Takeaways + What Transfers to Robotics

### What We Built
- Real-time weather radar with **smooth animation**, **3D volume rendering**, and **motion-compensated interpolation**
- End-to-end: NOAA S3 bucket to interactive browser visualization in **~90 seconds** cold start
- Pure Python backend (7 pip deps, 0 C deps), ~37 KB frontend

### What Transfers to Robotics

| Concept | Weather Radar | Robotics Analog |
|---|---|---|
| **Sparse data storage** | 95-99% empty grid, CSR matrices | Point clouds, occupancy grids |
| **Sensor fusion** | 180+ radars into one product | Multi-sensor fusion (lidar + camera + IMU) |
| **GPU-side processing** | GLSL shaders for colorization + compositing | CUDA kernels for perception pipelines |
| **Motion estimation** | FFT block matching between frames | Optical flow, scene flow, ICP registration |
| **Real-time streaming** | 2-min update cadence with interpolation | Sensor streams with prediction between updates |
| **Custom binary decoders** | GRIB2 parser (~400 lines) | ROS bag readers, proprietary sensor protocols |
| **Tile-based LOD** | TMS zoom levels for progressive detail | Octrees, voxel LOD for 3D maps |

### Try It
```bash
pip install -r backend/requirements.txt
uvicorn backend.main:app     # http://localhost:8000
```

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

stats = [
    ('180+', 'radars fused', ACCENT, 0),
    ('2 min', 'update cadence', ACCENT2, 1),
    ('24.5M', 'grid points / frame', ACCENT3, 2),
    ('20x', 'memory savings (sparse)', '#b39ddb', 3),
    ('~400', 'lines of GRIB2 decoder', ACCENT, 4),
    ('37 KB', 'frontend bundle', ACCENT2, 5),
    ('7', 'pip dependencies (0 C)', ACCENT3, 6),
    ('3', 'GPU view modes', '#b39ddb', 7),
]

cols = 4
for i, (number, label, color, idx) in enumerate(stats):
    col = idx % cols
    row = idx // cols
    x = 1.5 + col * 3.2
    y = 5.5 - row * 3.2

    # Circle background
    circle = plt.Circle((x, y + 0.3), 1.1, color=color, alpha=0.08)
    ax.add_patch(circle)
    circle_border = plt.Circle((x, y + 0.3), 1.1, color=color, alpha=0.3, fill=False, linewidth=2)
    ax.add_patch(circle_border)

    # Number
    ax.text(x, y + 0.5, number, ha='center', va='center',
            fontsize=26, fontweight='bold', color=color)

    # Label below
    ax.text(x, y - 1, label, ha='center', va='center',
            fontsize=10, color=DIM)

ax.set_title('Project by the Numbers', fontsize=18, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()